# ML Practical 3 — Diabetes Prediction using K-Nearest Neighbors (KNN)

**Objective:** Implement the KNN algorithm on the Diabetes dataset and evaluate using:  
Confusion Matrix, Accuracy, Error Rate, Precision, and Recall.  
**Dataset:** diabetes.csv — place the file downloaded from Kaggle in this folder.  
**Source:** https://www.kaggle.com/datasets/abdallamahgoub/diabetes


---
## Step 1 — Import Libraries

We import:
- `pandas` and `numpy` for data handling
- `StandardScaler` to normalize feature values before KNN (distance-sensitive)
- `KNeighborsClassifier` for the KNN model
- Metrics from `sklearn.metrics` to evaluate model performance
- `matplotlib` and `seaborn` for plots

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    confusion_matrix, accuracy_score,
    precision_score, recall_score,
    classification_report
)

print('Libraries imported successfully')

---
## Step 2 — Load the Dataset

We read `diabetes.csv` into a DataFrame.  
The dataset has 768 rows and 9 columns — 8 medical features and 1 target column (`Outcome`).  
`head()` shows the first 5 rows for a quick look at the data.

In [ ]:
# Place diabetes.csv (from Kaggle) in this folder
df = pd.read_csv('diabetes.csv')

print('Shape:', df.shape)
df.head()

---
## Step 3 — Explore the Dataset

`describe()` gives us statistical summaries — min, max, mean, std for every column.  
This helps spot potential issues: for example, zero values in `Glucose`, `BMI`, or `BloodPressure` are biologically impossible and indicate missing data encoded as 0.

In [ ]:
df.describe()

In [ ]:
print('Missing values:', df.isnull().sum().sum())
print('\nClass distribution:')
print(df['Outcome'].value_counts())

df['Outcome'].value_counts().plot(kind='bar', color=['steelblue', 'salmon'])
plt.title('Outcome Distribution (0 = No Diabetes, 1 = Diabetes)')
plt.xlabel('Outcome')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.show()

---
## Step 4 — Handle Zero Values (Data Cleaning)

Columns like `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, and `BMI` cannot be zero in reality.  
We replace these zeros with the **column median** — median is preferred over mean because it is less affected by outliers.

In [ ]:
# These columns cannot have 0 as a valid value
cols_with_zeros = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

for col in cols_with_zeros:
    median_val = df[col].median()
    df[col] = df[col].replace(0, median_val)

print('Zero values replaced with column median.')
df.describe()

---
## Step 5 — Separate Features and Target

- **X** = all 8 medical feature columns (Pregnancies, Glucose, BloodPressure, etc.)
- **y** = `Outcome` column (0 = No Diabetes, 1 = Diabetes)

In [ ]:
X = df.drop(columns=['Outcome'])
y = df['Outcome']

print('Features:', X.columns.tolist())
print('Features shape:', X.shape)
print('Target shape  :', y.shape)

---
## Step 6 — Split into Training and Test Sets

We use an 80/20 split — 80% for training, 20% for testing.  
`stratify=y` ensures the ratio of diabetic to non-diabetic patients is the same in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Training samples:', X_train.shape[0])
print('Testing  samples:', X_test.shape[0])

---
## Step 7 — Normalize the Data

KNN calculates Euclidean distance between data points.  
If one feature (e.g., Insulin ~80) has a much larger range than another (e.g., Pregnancies ~0-17), it will dominate the distance calculation.  
`StandardScaler` brings all features to the same scale (mean=0, std=1), ensuring fair distance comparisons.

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)   # fit on train only
X_test  = scaler.transform(X_test)         # apply same scale to test

print('Data normalized.')

---
## Step 8 — Find the Optimal K

We test K values from 1 to 20 and record accuracy for each.  
Plotting this helps us choose the K that gives the best test accuracy without overfitting (K=1 would perfectly fit training data but generalize poorly).

In [ ]:
k_values   = range(1, 21)
accuracies = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    accuracies.append(accuracy_score(y_test, knn.predict(X_test)))

best_k = k_values[accuracies.index(max(accuracies))]
print(f'Best K: {best_k}  |  Accuracy: {max(accuracies)*100:.2f}%')

plt.figure(figsize=(10, 4))
plt.plot(k_values, accuracies, marker='o', color='steelblue')
plt.axvline(x=best_k, color='red', linestyle='--', label=f'Best K={best_k}')
plt.title('KNN Accuracy vs K')
plt.xlabel('K (Number of Neighbors)')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

---
## Step 9 — Train the Final KNN Model

We train the KNN classifier using the best K found above and make predictions on the test set.

In [ ]:
knn = KNeighborsClassifier(n_neighbors=best_k)
knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)

print(f'Model trained with K={best_k}')
print('Predictions done on test set.')

---
## Step 10 — Confusion Matrix

A confusion matrix shows 4 outcomes:
- **True Negative (TN):** Correctly predicted as No Diabetes
- **False Positive (FP):** Predicted Diabetes but actually No Diabetes
- **False Negative (FN):** Predicted No Diabetes but actually has Diabetes
- **True Positive (TP):** Correctly predicted as Diabetes

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Diabetes', 'Diabetes'],
            yticklabels=['No Diabetes', 'Diabetes'])
plt.title(f'Confusion Matrix (K={best_k})')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

TN, FP, FN, TP = cm.ravel()
print(f'True Negatives  (TN): {TN}')
print(f'False Positives (FP): {FP}')
print(f'False Negatives (FN): {FN}')
print(f'True Positives  (TP): {TP}')

---
## Step 11 — Accuracy, Error Rate, Precision, Recall

Formulas:
- **Accuracy** = (TP + TN) / Total — proportion of correct predictions
- **Error Rate** = 1 - Accuracy — proportion of wrong predictions
- **Precision** = TP / (TP + FP) — of all predicted diabetic, how many actually are?
- **Recall** = TP / (TP + FN) — of all actual diabetic patients, how many did we catch?

In [ ]:
accuracy   = accuracy_score(y_test, y_pred)
error_rate = 1 - accuracy
precision  = precision_score(y_test, y_pred)
recall     = recall_score(y_test, y_pred)

print('=' * 40)
print(f'  Accuracy   : {accuracy   * 100:.2f}%')
print(f'  Error Rate : {error_rate * 100:.2f}%')
print(f'  Precision  : {precision  * 100:.2f}%')
print(f'  Recall     : {recall     * 100:.2f}%')
print('=' * 40)

In [ ]:
# Full classification report
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['No Diabetes', 'Diabetes']))

---
## Step 12 — Visualize All Metrics

A bar chart to visually compare the four evaluation metrics at once.

In [ ]:
metrics = ['Accuracy', 'Error Rate', 'Precision', 'Recall']
values  = [accuracy, error_rate, precision, recall]
colors  = ['steelblue', 'salmon', 'seagreen', 'mediumpurple']

plt.figure(figsize=(8, 4))
bars = plt.bar(metrics, values, color=colors, width=0.5)
for bar, val in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val*100:.2f}%', ha='center', fontsize=10)
plt.ylim(0, 1.15)
plt.title(f'KNN Performance Metrics (K={best_k})')
plt.ylabel('Score')
plt.show()

---
## Summary

| Step | What we did |
|------|-------------|
| 1 | Imported libraries |
| 2 | Loaded diabetes.csv |
| 3 | Explored data with describe() and class distribution |
| 4 | Replaced invalid zero values with column median |
| 5 | Separated features (X) and target (y) |
| 6 | 80/20 train-test split with stratification |
| 7 | Normalized with StandardScaler |
| 8 | Found optimal K by testing K=1 to K=20 |
| 9 | Trained final KNN model with best K |
| 10 | Plotted confusion matrix |
| 11 | Computed Accuracy, Error Rate, Precision, Recall |
| 12 | Visualized all metrics in a bar chart |

**Metric Interpretations:**
- **Accuracy** — overall correctness of the model
- **Error Rate** — complement of accuracy; how often the model is wrong
- **Precision** — avoids false alarms (falsely flagging healthy people as diabetic)
- **Recall** — avoids missed detections (missing actual diabetic patients) — more critical in medical diagnosis
